# Stock Triple-Barrier Classification · 01 Data Understanding

Kaggle version of [dl-multiclassification-stock-triple-barrier-rnn-lstm](https://github.com/Pisit-Janthawee/dl-multiclassification-stock-triple-barrier-rnn-lstm): the 5 repo notebooks in one notebook, with the `src/` package written to disk in [0/5].

**Run on Kaggle** → Settings: Accelerator **GPU T4**, Internet **On** (prices come from Yahoo Finance via `yfinance`) → Run All. No dataset to add.

# [0/5] Kaggle setup

Install what Kaggle lacks and write `src/` → every `from src... import` below works unchanged

In [ ]:
"""Optional · Kaggle setup. Keras runs on the PyTorch backend, as in the repo."""
%pip install -q yfinance==1.2.0
import os
os.environ['KERAS_BACKEND'] = 'torch'

In [ ]:
import os
os.makedirs('src', exist_ok=True)

In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
"""Project paths and constants shared by every notebook."""
from pathlib import Path

RANDOM_STATE = 42

# ---------------------------------------------------------------- Universe & period
TICKERS = ["NVDA", "CRWD", "AMD", "ADBE", "MSFT", "GOOGL",            # IT stocks of my first version
           "INTC", "AAPL", "AMZN", "CRM", "IBM", "TSLA", "PYPL", "XYZ"]  # XYZ = Block (ticker SQ until 2025)
MARKET = "SPY"                     # market context (S&P 500 ETF)
START = "2010-01-01"
END = "2026-11-12"                 # fixed end date -> reproducible downloads (exclusive in yfinance)

# ---------------------------------------------------------------- Triple-barrier settings
VOL_SPAN = 20                      # EWM span (days) of the daily volatility estimate
PT_MULT, SL_MULT = 0.75, 0.75      # barrier = mult x daily vol x sqrt(horizon); 0.75 gives balanced classes
HORIZON = 10                       # vertical barrier (trading days)
CLASSES = {-1: "stop-loss", 0: "time-out", 1: "take-profit"}

# ---------------------------------------------------------------- Time split (by event start date)
VALID_START = "2022-01-01"
TEST_START = "2024-01-01"

# ---------------------------------------------------------------- Paths
ROOT = Path(__file__).resolve().parents[1]
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
for _d in (RAW_DIR, PROCESSED_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- Colours
LABEL_COLORS = {-1: "#dc2626", 0: "#9ca3af", 1: "#15803d"}   # stop-loss red, time-out grey, take-profit green
PRICE_COLOR = "#374151"


In [ ]:
%%writefile src/data.py
"""Download daily prices with yfinance and cache them as Parquet (data/raw is git-ignored)."""
import pandas as pd

from .config import END, MARKET, RAW_DIR, START, TICKERS

RAW_FILE = RAW_DIR / "prices.parquet"


def download_prices(tickers=None, start=START, end=END, refresh=False) -> pd.DataFrame:
    """Long table: date, ticker, open, high, low, close, volume (split/dividend adjusted).

    Cached in data/raw/prices.parquet; set refresh=True to download again.
    """
    if RAW_FILE.exists() and not refresh:
        return pd.read_parquet(RAW_FILE)
    import yfinance as yf

    tickers = tickers or TICKERS + [MARKET]
    wide = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False, group_by="column")
    long = (wide.stack(level="Ticker", future_stack=True)
                .rename_axis(["date", "ticker"]).reset_index())
    long.columns = [c.lower() for c in long.columns]
    long = long.dropna(subset=["close"]).sort_values(["ticker", "date"]).reset_index(drop=True)
    long = long[["date", "ticker", "open", "high", "low", "close", "volume"]]
    long.to_parquet(RAW_FILE, index=False)
    return long


def save_processed(df: pd.DataFrame, name: str) -> None:
    from .config import PROCESSED_DIR
    df.to_parquet(PROCESSED_DIR / f"{name}.parquet", index=False)


def load_processed(name: str) -> pd.DataFrame:
    from .config import PROCESSED_DIR
    return pd.read_parquet(PROCESSED_DIR / f"{name}.parquet")


In [ ]:
%%writefile src/evaluate.py
"""Splitting, sequence building, metrics and the backtest."""
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, f1_score, log_loss

LABELS = [-1, 0, 1]


# ---------------------------------------------------------------- Split with purging
def assign_split(df: pd.DataFrame, valid_start: str, test_start: str) -> pd.Series:
    """Split by event start date, then PURGE events whose label end t1 reaches into the next set
    (their outcome would overlap the next period's prices)."""
    vs, ts = pd.Timestamp(valid_start), pd.Timestamp(test_start)
    split = np.select([df["date"] < vs, df["date"] < ts], ["train", "valid"], "test")
    purge = ((split == "train") & (df["t1"] >= vs)) | ((split == "valid") & (df["t1"] >= ts))
    return pd.Series(np.where(purge, "purged", split), index=df.index)


class PurgedKFold:
    """K-fold over time-ordered events with purging and an embargo (Lopez de Prado, ch. 7).

    For each test fold [a, b]:
      - purge: drop training events whose interval [t0, t1] overlaps the test period
      - embargo: also drop training events starting within `embargo` days after the test period
    """

    def __init__(self, n_splits=5, embargo_days=5):
        self.n_splits, self.embargo = n_splits, pd.Timedelta(days=embargo_days)

    def split(self, t0: pd.Series, t1: pd.Series):
        dates = np.sort(t0.unique())
        for chunk in np.array_split(dates, self.n_splits):
            a, b = chunk[0], chunk[-1]
            test = (t0 >= a) & (t0 <= b)
            overlap = (t0 <= b) & (t1 >= a)
            embargo = (t0 > b) & (t0 <= b + self.embargo)
            train = ~test & ~overlap & ~embargo
            yield np.flatnonzero(train.to_numpy()), np.flatnonzero(test.to_numpy())


# ---------------------------------------------------------------- Sequences for RNN / LSTM / GRU
def make_sequences(df: pd.DataFrame, features: list, lookback: int):
    """For each event row, the window of the last `lookback` days of features of the same ticker
    (ending at the event day). Rows without a full window are dropped.
    Returns X (n, lookback, n_features) and the row index it belongs to."""
    X, idx = [], []
    for _, g in df.groupby("ticker", sort=False):
        values = g[features].to_numpy(dtype=np.float32)
        for j in range(lookback - 1, len(g)):
            X.append(values[j - lookback + 1: j + 1])
            idx.append(g.index[j])
    return np.stack(X), np.array(idx)


# ---------------------------------------------------------------- Metrics
def score(y_true, proba, name="model", weights=None) -> dict:
    """proba columns ordered as LABELS (-1, 0, 1)."""
    pred = np.array(LABELS)[proba.argmax(1)]
    return {
        "model": name,
        "macro_f1": f1_score(y_true, pred, average="macro"),
        "balanced_acc": balanced_accuracy_score(y_true, pred),
        "log_loss": log_loss(y_true, proba, labels=LABELS, sample_weight=weights),
        "tp_precision": ((pred == 1) & (y_true == 1)).sum() / max((pred == 1).sum(), 1),
        "tp_recall": ((pred == 1) & (y_true == 1)).sum() / max((y_true == 1).sum(), 1),
    }


# ---------------------------------------------------------------- Backtest
def backtest_long(events: pd.DataFrame, signal: pd.Series, prices: pd.DataFrame, cost_bps: float = 10.0):
    """Long-only, one position per ticker at a time.

    Enter at the close of the event day when `signal` is True and the ticker is flat; exit at the close
    of t1 (barrier touch or vertical barrier). `cost_bps` is charged on entry and on exit.
    Returns (daily portfolio returns, trades). Portfolio = equal weight across tickers, idle cash earns 0.
    """
    trades, daily = [], {}
    close = prices.pivot(index="date", columns="ticker", values="close")
    rets = np.log(close).diff()
    cost = cost_bps / 1e4
    for ticker, g in events[signal.reindex(events.index).fillna(False).to_numpy()].groupby("ticker"):
        held = pd.Series(0.0, index=close.index)
        busy_until = pd.Timestamp.min
        for _, e in g.sort_values("date").iterrows():
            if e["date"] <= busy_until:
                continue
            span = (close.index > e["date"]) & (close.index <= e["t1"])
            held[span] = rets.loc[span, ticker]
            first, last = np.flatnonzero(span)[[0, -1]]
            held.iloc[first] -= cost
            held.iloc[last] -= cost
            trades.append({"ticker": ticker, "entry": e["date"], "exit": e["t1"], "label": e["label"],
                           "ret": held[span].sum()})
            busy_until = e["t1"]
        daily[ticker] = held
    daily = pd.DataFrame(daily).reindex(columns=events["ticker"].unique()).fillna(0.0)
    return daily, pd.DataFrame(trades)


def performance(log_returns: pd.Series, periods: int = 252) -> dict:
    """CAGR, annualised volatility, Sharpe (rf = 0) and maximum drawdown of a daily log-return series."""
    r = log_returns.dropna()
    equity = np.exp(r.cumsum())
    years = len(r) / periods
    simple = np.exp(r) - 1
    return {
        "total_return": equity.iloc[-1] - 1,
        "cagr": equity.iloc[-1] ** (1 / years) - 1,
        "ann_vol": simple.std() * np.sqrt(periods),
        "sharpe": simple.mean() / simple.std() * np.sqrt(periods) if simple.std() > 0 else np.nan,
        "max_drawdown": (equity / equity.cummax() - 1).min(),
    }


In [ ]:
%%writefile src/features.py
"""Stationary, causal features (each value at day t uses data up to and including day t only)."""
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller


# ---------------------------------------------------------------- Fractional differentiation
def ffd_weights(d: float, threshold: float = 1e-4, max_len: int = 500) -> np.ndarray:
    """Weights of the fixed-width window fractional difference (1 - B)^d.
    w_0 = 1, w_k = -w_{k-1} * (d - k + 1) / k ; stop when |w_k| < threshold."""
    w = [1.0]
    for k in range(1, max_len):
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < threshold:
            break
        w.append(w_k)
    return np.array(w)


def frac_diff(series: pd.Series, d: float, threshold: float = 1e-3) -> pd.Series:
    """x~_t = sum_k w_k x_{t-k}  (NaN for the first len(w)-1 values)."""
    w = ffd_weights(d, threshold)
    x = series.to_numpy(dtype=float)
    out = np.full(len(x), np.nan)
    if len(x) >= len(w):
        out[len(w) - 1:] = np.convolve(x, w, mode="valid")
    return pd.Series(out, index=series.index)


def min_d_stationary(log_price: pd.Series, d_grid=np.arange(0, 1.01, 0.05), alpha=0.05, threshold=1e-3):
    """Smallest d whose fractionally differenced series passes the ADF test (keeps the most memory).
    Returns (d, table of ADF p-value and correlation with the original series per d)."""
    rows = []
    for d in d_grid:
        fd = frac_diff(log_price, d, threshold).dropna()
        p = adfuller(fd, maxlag=1, regression="c", autolag=None)[1]
        corr = np.corrcoef(log_price.loc[fd.index], fd)[0, 1]
        rows.append({"d": round(d, 2), "adf_p": p, "corr_with_price": corr})
    table = pd.DataFrame(rows)
    ok = table[table.adf_p < alpha]
    return (float(ok.d.iloc[0]) if len(ok) else 1.0), table


# ---------------------------------------------------------------- Technical indicators
def rsi(close: pd.Series, n: int = 14) -> pd.Series:
    """Relative Strength Index (Wilder): 100 - 100 / (1 + avg_gain / avg_loss)."""
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / n, min_periods=n).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / n, min_periods=n).mean()
    return 100 - 100 / (1 + gain / loss)


def ticker_features(g: pd.DataFrame, market: pd.DataFrame, d: float) -> pd.DataFrame:
    """Feature table for ONE ticker. `g` and `market` have columns open/high/low/close/volume, date index."""
    c, h, l, v = g["close"], g["high"], g["low"], g["volume"]
    r = np.log(c).diff()
    f = pd.DataFrame(index=g.index)
    for k in (1, 5, 10, 21):
        f[f"ret_{k}"] = np.log(c / c.shift(k))
    f["vol_21"] = r.rolling(21).std()
    f["vol_ratio_5_21"] = r.rolling(5).std() / f["vol_21"]
    f["rsi_14"] = rsi(c, 14) / 100
    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    f["macd_hist"] = (macd - macd.ewm(span=9, adjust=False).mean()) / c
    sma20, std20 = c.rolling(20).mean(), c.rolling(20).std()
    f["bb_pctb"] = (c - (sma20 - 2 * std20)) / (4 * std20)
    f["bb_width"] = 4 * std20 / sma20
    f["dist_sma_50"] = c / c.rolling(50).mean() - 1
    f["dist_sma_200"] = c / c.rolling(200).mean() - 1
    f["dist_ema_21"] = c / c.ewm(span=21, adjust=False).mean() - 1
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    f["atr_14"] = tr.ewm(alpha=1 / 14, min_periods=14).mean() / c
    logv = np.log1p(v)
    f["volume_z"] = (logv - logv.rolling(21).mean()) / logv.rolling(21).std()
    f["ffd_close"] = frac_diff(np.log(c), d)
    # market context
    mc = market["close"].reindex(g.index)
    mr = np.log(mc).diff()
    f["mkt_ret_1"] = mr
    f["mkt_ret_5"] = np.log(mc / mc.shift(5))
    f["mkt_vol_21"] = mr.rolling(21).std()
    f["rel_strength_21"] = f["ret_21"] - np.log(mc / mc.shift(21))
    f["beta_63"] = r.rolling(63).cov(mr) / mr.rolling(63).var()
    return f.replace([np.inf, -np.inf], np.nan)


FEATURES = ["ret_1", "ret_5", "ret_10", "ret_21", "vol_21", "vol_ratio_5_21", "rsi_14", "macd_hist",
            "bb_pctb", "bb_width", "dist_sma_50", "dist_sma_200", "dist_ema_21", "atr_14", "volume_z",
            "ffd_close", "mkt_ret_1", "mkt_ret_5", "mkt_vol_21", "rel_strength_21", "beta_63"]


In [ ]:
%%writefile src/labeling.py
"""Triple-barrier labeling, sample uniqueness and the CUSUM filter (Lopez de Prado, *Advances in Financial ML*, ch. 2-4).

For an event starting at day t0 with close P0 and daily volatility s:
    upper barrier  = P0 * exp(+pt_mult * s * sqrt(horizon))   -> label +1 (take-profit)
    lower barrier  = P0 * exp(-sl_mult * s * sqrt(horizon))   -> label -1 (stop-loss)
    vertical       = t0 + horizon trading days                -> label  0 (time-out)
The label is the barrier touched FIRST by the closing price on days t0+1 ... t0+horizon.
"""
import numpy as np
import pandas as pd


def daily_volatility(close: pd.Series, span: int = 20) -> pd.Series:
    """EWM standard deviation of daily log returns (uses data up to and including t)."""
    return np.log(close).diff().ewm(span=span, min_periods=span).std()


def triple_barrier(close: pd.Series, vol: pd.Series, horizon: int = 10,
                   pt_mult: float = 2.0, sl_mult: float = 2.0) -> pd.DataFrame:
    """Label every day of ONE ticker. Returns a DataFrame indexed like `close` with:
    label (-1/0/1), t1 (date the label is known = touch date or vertical barrier),
    days_held, ret (log return from t0 to t1), barrier (width in log-return units)."""
    logp = np.log(close.to_numpy())
    v = vol.to_numpy()
    dates = close.index
    n = len(close)
    label = np.full(n, np.nan)
    t1_idx = np.full(n, -1)
    ret = np.full(n, np.nan)
    width = v * np.sqrt(horizon)
    for i in range(n - horizon):            # last `horizon` days have no complete future -> unlabeled
        if np.isnan(width[i]):
            continue
        path = logp[i + 1: i + 1 + horizon] - logp[i]
        up = np.flatnonzero(path >= pt_mult * width[i])
        dn = np.flatnonzero(path <= -sl_mult * width[i])
        first_up = up[0] if len(up) else horizon
        first_dn = dn[0] if len(dn) else horizon
        if first_up < first_dn:
            label[i], k = 1, first_up
        elif first_dn < first_up:
            label[i], k = -1, first_dn
        else:                                 # neither barrier touched (equal only when both == horizon)
            label[i], k = 0, horizon - 1
        t1_idx[i] = i + 1 + k
        ret[i] = path[k]
    out = pd.DataFrame({"label": label, "ret": ret, "barrier": width}, index=dates)
    valid = t1_idx >= 0
    out["t1"] = pd.NaT
    out.loc[valid, "t1"] = dates[t1_idx[valid]]
    out["days_held"] = np.where(valid, t1_idx - np.arange(n), np.nan)
    return out


def average_uniqueness(t0: pd.Index, t1: pd.Series, all_dates: pd.Index) -> pd.Series:
    """Average uniqueness of each event (one ticker).

    Concurrency c_t = number of events whose holding interval (t0, t1] contains day t.
    Uniqueness of event i = mean over its interval of 1 / c_t  (1 = no overlap, 1/H = fully overlapped).
    """
    pos = pd.Series(np.arange(len(all_dates)), index=all_dates)
    start = pos.reindex(t0).to_numpy() + 1          # holding starts the day after t0
    end = pos.reindex(t1).to_numpy()
    conc = np.zeros(len(all_dates) + 1)
    np.add.at(conc, start, 1)
    np.add.at(conc, end + 1, -1)
    conc = np.cumsum(conc)[:-1]
    inv = np.where(conc > 0, 1 / np.maximum(conc, 1), 0)
    csum = np.concatenate([[0], np.cumsum(inv)])
    u = (csum[end + 1] - csum[start]) / (end - start + 1)
    return pd.Series(u, index=t0)


def cusum_filter(close: pd.Series, threshold: pd.Series | float) -> pd.DatetimeIndex:
    """Symmetric CUSUM filter on log returns: emit an event when the cumulative up- or down-move
    since the last event exceeds `threshold` (a number or a per-day Series)."""
    r = np.log(close).diff().dropna()
    h = threshold.reindex(r.index) if isinstance(threshold, pd.Series) else pd.Series(threshold, index=r.index)
    events, s_pos, s_neg = [], 0.0, 0.0
    for t, x in r.items():
        if np.isnan(h[t]):
            continue
        s_pos, s_neg = max(0.0, s_pos + x), min(0.0, s_neg + x)
        if s_pos > h[t]:
            s_pos = 0.0
            events.append(t)
        elif s_neg < -h[t]:
            s_neg = 0.0
            events.append(t)
    return pd.DatetimeIndex(events)


def label_all(prices: pd.DataFrame, vol_span=20, horizon=10, pt_mult=2.0, sl_mult=2.0) -> pd.DataFrame:
    """Triple-barrier labels + uniqueness for every ticker in a long price table."""
    parts = []
    for ticker, g in prices.groupby("ticker", sort=True):
        close = g.set_index("date")["close"]
        vol = daily_volatility(close, vol_span)
        lab = triple_barrier(close, vol, horizon, pt_mult, sl_mult)
        lab["vol"] = vol
        lab = lab.dropna(subset=["label"])
        lab["uniqueness"] = average_uniqueness(lab.index, lab["t1"], close.index).to_numpy()
        lab["ticker"] = ticker
        parts.append(lab.rename_axis("date").reset_index())
    out = pd.concat(parts, ignore_index=True)
    out["label"] = out["label"].astype(int)
    return out


In [ ]:
%%writefile src/plotting.py
"""Shared plot style + helper to save figures used in the README."""
import matplotlib.pyplot as plt
import seaborn as sns

from .config import FIGURES_DIR

CMAP = "Greys"


def set_style():
    """Grey 'darkgrid' look (same as my Ensemble / Tuberculosis notebooks)."""
    sns.set_style("darkgrid")
    sns.set_palette("gray", 8)
    sns.set_context("notebook", font_scale=1.05)
    plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold"})


def savefig(fig, name: str):
    """Save into reports/figures/<name>.png (these images are committed)."""
    fig.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight", dpi=110)


# Part [1/5] 01_data_understanding

Repo notebook → [notebooks/01_data_understanding.ipynb](https://github.com/Pisit-Janthawee/dl-multiclassification-stock-triple-barrier-rnn-lstm/blob/main/notebooks/01_data_understanding.ipynb)

# Stock Triple-Barrier Classification · 01 Data Understanding

# Aim, Objective, and Motivation
- **Aim:** <u>understand</u> how daily stock prices behave before trying to predict anything.
- **Objective:** check data health, compare prices with returns (stationarity, fat tails, volatility clustering)
  and show why my first version's high R² was misleading.
- **Motivation:** my first version (RNN / LSTM / GRU) predicted the **scaled closing price** and reached R² ≈ 0.99.
  This project replaces that target with **triple-barrier labels**: will a trade starting today hit its
  take-profit, its stop-loss, or run out of time?

**Data:** daily OHLCV from Yahoo Finance via `yfinance` (split and dividend adjusted), 2010-01-01 to 2026-11-11,
for the 14 stocks of my first version plus `SPY` (S&P 500 ETF) as market context.

# Outlines
1. Download prices
2. Data health
3. Prices over time
4. Returns: distribution and fat tails
5. Stationarity: price vs return
6. Autocorrelation and volatility clustering
7. The price-level trap
    - 7.1 Naive "tomorrow = today" forecast
    - 7.2 Scaling before splitting
8. Summary & decisions

In [ ]:
# ---------------------------------------------------------------- Imports
import sys
from pathlib import Path


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller

from src.config import TICKERS, MARKET, START, END, PRICE_COLOR
from src.data import download_prices
from src.plotting import set_style, savefig

# ---------------------------------------------------------------- Global constants
EXAMPLE = "AAPL"                  # ticker used for single-stock illustrations
ACF_LAGS = 40
TEST_SHARE = 0.2                  # last 20 % of days as "test" in the price-level trap demo (as in my first version)
ALPHA = 0.05                      # significance level for statistical tests

set_style()

# [1/8] Download prices
> **Objective:** <u>download</u> adjusted daily prices once and cache them in `data/raw/prices.parquet` (git-ignored).
> `END` is fixed, so re-running gives the same data (Yahoo can still revise adjusted prices after new splits or dividends).

In [ ]:
prices = download_prices()
close = prices.pivot(index="date", columns="ticker", values="close")

In [ ]:
print(f"{len(prices):,} rows | {prices.ticker.nunique()} tickers | {prices.date.min().date()} to {prices.date.max().date()}")
prices.head()

# [2/8] Data health
> **Question:** are there missing days, gaps or impossible values?

In [ ]:
health = prices.groupby("ticker").agg(
    first_day=("date", "min"), last_day=("date", "max"), trading_days=("date", "size"),
    missing_close=("close", lambda s: s.isna().sum()),
    zero_volume_days=("volume", lambda s: (s == 0).sum()),
    non_positive_price=("close", lambda s: (s <= 0).sum()),
)
largest_moves = (np.log(close).diff().abs().max().sort_values(ascending=False) * 100)

In [ ]:
health

In [ ]:
largest_moves.round(1).rename("largest |daily log return| %").head(6)

**Answer:** the data is clean: no missing closes and no zero or negative prices (one day with zero volume).
Four stocks start later because they listed later: `TSLA` (2010-06), `PYPL` (2015-07), `XYZ` (2015-11, ticker `SQ` until 2025)
and `CRWD` (2019-06). The largest daily moves (up to about 40 %) are real earnings or market shocks, not errors, so nothing is removed.

# [3/8] Prices over time
> **Objective:** <u>compare</u> stocks on one scale: growth of \$1 invested at each stock's first day.

In [ ]:
growth = close.apply(lambda s: s / s.dropna().iloc[0])
final_growth = growth.ffill().iloc[-1].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
for t in growth.columns:
    lw, c = (2.2, "#dc2626") if t == MARKET else (1, None)
    ax.plot(growth.index, growth[t], lw=lw, color=c, label=t)
ax.set(yscale="log", ylabel="growth of $1 (log scale)", title="Growth of $1 since each stock's first day")
ax.legend(ncol=5, fontsize=8, loc="upper left")
savefig(fig, "01_growth")
plt.close(fig)

In [ ]:
final_growth.round(1).rename("growth of $1")

In [ ]:
fig

**Answer:** growth ranges from ×411 (`NVDA`) and ×245 (`TSLA`) down to ×1.6 (`PYPL`); the market (`SPY`) grew ×9. (!) This universe was picked **today** and is
dominated by famous winners: that is **survivorship / selection bias**, and it flatters any long-only strategy. The backtest in
notebook 05 therefore compares against buy-and-hold of the **same** stocks and against random entries.

# [4/8] Returns: distribution and fat tails
> **Question:** are daily returns normally distributed?
>
> **Formulas:** log return $r_t = \ln\frac{P_t}{P_{t-1}}$ (additive over time) ·
> excess kurtosis $= \frac{1}{n}\sum\left(\frac{r_t - \bar{r}}{s}\right)^4 - 3$ (0 for a normal distribution) ·
> Jarque-Bera $JB = \frac{n}{6}\left(S^2 + \frac{K^2}{4}\right)$, $H_0$: returns are normal.

In [ ]:
returns = np.log(close).diff()
moments = pd.DataFrame({
    "mean_daily_%": returns.mean() * 100,
    "std_daily_%": returns.std() * 100,
    "skew": returns.skew(),
    "excess_kurtosis": returns.kurt(),
    "jarque_bera_p": returns.apply(lambda s: stats.jarque_bera(s.dropna()).pvalue),
}).sort_values("std_daily_%", ascending=False)

r = returns[EXAMPLE].dropna()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.histplot(r, bins=150, stat="density", color=PRICE_COLOR, ax=axes[0], label="observed")
x = np.linspace(r.min(), r.max(), 400)
axes[0].plot(x, stats.norm.pdf(x, r.mean(), r.std()), color="#dc2626", lw=2, label="normal, same mean & std")
axes[0].set(xlim=(-0.1, 0.1), title=f"{EXAMPLE} daily log returns", xlabel="log return")
axes[0].legend()
stats.probplot(r, dist="norm", plot=axes[1])
axes[1].get_lines()[0].set(color=PRICE_COLOR, markersize=3)
axes[1].get_lines()[1].set(color="#dc2626")
axes[1].set_title(f"{EXAMPLE} Q-Q plot vs normal")
savefig(fig, "01_returns_distribution")
plt.close(fig)

In [ ]:
moments.round(3)

In [ ]:
fig

**Answer:** no. Every stock has positive excess kurtosis (fat tails) and Jarque-Bera rejects normality with p ≈ 0.
Extreme days happen far more often than a normal distribution predicts (the Q-Q plot bends away at both ends).
→ Barriers should scale with each stock's **own volatility**, and a fixed percentage threshold would mean very different
things for `IBM` (≈ 1.6 % daily std) and `TSLA` (≈ 3.6 %).

# [5/8] Stationarity: price vs return
> **Question:** do prices and returns have a stable mean and variance over time?
>
> **Augmented Dickey-Fuller test:** $\Delta y_t = \alpha + \gamma\, y_{t-1} + \sum_{i} \beta_i\, \Delta y_{t-i} + \varepsilon_t$,
> $H_0$: $\gamma = 0$ (unit root, non-stationary). p < 0.05 → stationary.

In [ ]:
adf = pd.DataFrame({
    "adf_p_log_price": np.log(close).apply(lambda s: adfuller(s.dropna(), maxlag=1, autolag=None)[1]),
    "adf_p_log_return": returns.apply(lambda s: adfuller(s.dropna(), maxlag=1, autolag=None)[1]),
})
adf["price_stationary"] = adf["adf_p_log_price"] < ALPHA
adf["return_stationary"] = adf["adf_p_log_return"] < ALPHA

In [ ]:
adf.round(4)

**Answer:** log prices are **non-stationary** for every stock (ADF p between 0.31 and 0.99, unit root not rejected), while
returns are stationary for all of them (p ≈ 0). Models trained on price levels learn "the price is near where it was", which does not transfer to new price
ranges. → Features in this project are built from **returns and ratios**, plus a *fractionally* differenced price that keeps
some memory while being stationary (notebook 03).

# [6/8] Autocorrelation and volatility clustering
> **Question:** do past returns predict future returns? Does past volatility predict future volatility?
>
> **Autocorrelation at lag k:** $\rho_k = \frac{\sum_t (r_t - \bar r)(r_{t-k} - \bar r)}{\sum_t (r_t - \bar r)^2}$;
> the band is the 95 % confidence interval of zero autocorrelation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
plot_acf(r, lags=ACF_LAGS, ax=axes[0], color=PRICE_COLOR, vlines_kwargs={"colors": PRICE_COLOR})
axes[0].set(title=f"{EXAMPLE}: ACF of returns", ylim=(-0.15, 0.3))
plot_acf(r ** 2, lags=ACF_LAGS, ax=axes[1], color="#dc2626", vlines_kwargs={"colors": "#dc2626"})
axes[1].set(title=f"{EXAMPLE}: ACF of squared returns (volatility)", ylim=(-0.15, 0.3))
savefig(fig, "01_acf")
plt.close(fig)

acf_summary = pd.DataFrame({
    "acf1_returns": returns.apply(lambda s: s.autocorr(1)),
    "acf1_squared_returns": (returns ** 2).apply(lambda s: s.autocorr(1)),
})

In [ ]:
acf_summary.round(3)

In [ ]:
fig

**Answer:** returns have almost **no** autocorrelation (lag-1 between −0.10 and 0.05), so tomorrow's direction is very hard to predict
from past returns alone. Squared returns are positively autocorrelated for every stock (0.04 for `IBM` up to 0.40 for `SPY`):
**volatility clusters** (calm and turbulent periods persist).
→ Volatility is predictable, so barrier widths based on recent volatility are sensible; direction is where the difficulty lies.

# [7/8] The price-level trap
## 7.1 Naive "tomorrow = today" forecast
> **Question:** how good is R² ≈ 0.99 on scaled prices, really?
>
> **Formula:** $R^2 = 1 - \frac{\sum (y_t - \hat y_t)^2}{\sum (y_t - \bar y)^2}$. A naive forecast $\hat P_t = P_{t-1}$ needs no model at all.

In [ ]:
naive = pd.DataFrame({
    "r2_price_naive": close.apply(lambda s: r2_score(s.dropna()[1:], s.dropna().shift(1)[1:])),
    "r2_return_naive": returns.apply(lambda s: r2_score(s.dropna()[1:], s.dropna().shift(1)[1:])),
})

In [ ]:
naive.round(4)

**Answer:** copying yesterday's price gives **R² between 0.995 and 0.9996** on price levels, without learning anything, because
the variance of a trending price is huge compared with one day's change. On **returns** the same naive forecast scores about −1.
→ My first version's R² ≈ 0.99 mostly measured this effect; R² on prices is not evidence of forecasting skill.

## 7.2 Scaling before splitting
> **Question:** what goes wrong when `MinMaxScaler` is fitted on the full series before the train/test split (as in my first version)?

In [ ]:
s = close[EXAMPLE].dropna()
split = int(len(s) * (1 - TEST_SHARE))
leaky = MinMaxScaler().fit(s.to_frame())                     # fitted on train + test
honest = MinMaxScaler().fit(s.iloc[:split].to_frame())       # fitted on train only
scaled_test = pd.DataFrame({
    "leaky_scaler": leaky.transform(s.iloc[split:].to_frame()).ravel(),
    "train_only_scaler": honest.transform(s.iloc[split:].to_frame()).ravel(),
})

In [ ]:
scaled_test.describe().loc[["min", "max"]].round(2)

**Answer:** with the leaky scaler the test prices stay inside [0, 1] (0.45 to 1.00), because the scaler already saw the test period's
maximum. Fitted on training data only, test values reach 1.96 — the model would face prices it has never seen. The leaky version
quietly tells the model the future price range. → In this project every scaler is fitted on the **training period only**.

# [8/8] Summary & decisions
| Finding | Decision |
|---|---|
| Clean data; three stocks listed later | keep all; each stock contributes from its first day |
| Universe picked in hindsight (survivorship bias) | compare strategies with buy-and-hold of the same stocks and random entries |
| Fat tails, very different volatility per stock | barriers scaled by each stock's own recent volatility |
| Prices non-stationary, returns stationary | features from returns / ratios + fractionally differenced price |
| Returns barely autocorrelated, volatility clusters | volatility-based barriers; expect modest directional skill |
| R² on prices is ≈ 0.999 for a naive forecast | predict **triple-barrier outcomes** instead of price levels |
| Scaler fitted before splitting leaks the future | fit every transformation on the training period only |

# Part [2/5] 02_triple_barrier_labeling

Repo notebook → [notebooks/02_triple_barrier_labeling.ipynb](https://github.com/Pisit-Janthawee/dl-multiclassification-stock-triple-barrier-rnn-lstm/blob/main/notebooks/02_triple_barrier_labeling.ipynb)

# Stock Triple-Barrier Classification · 02 Triple-Barrier Labeling

# Aim, Objective, and Motivation
- **Aim:** <u>replace</u> "predict tomorrow's price" with a question a trader actually faces:
  *if I buy today, will the trade hit its profit target, its stop-loss, or run out of time first?*
- **Objective:** build triple-barrier labels for every stock and day, verify them, study how the settings change them,
  and handle the fact that neighbouring labels overlap in time.
- **Motivation:** fixed-horizon labels ("is the price higher in 10 days?") ignore the path: a trade that falls 15 % and then
  recovers gets the same label as one that rose steadily. Triple-barrier labels follow the path, like a real stop-loss order.

Method from Marcos López de Prado, *Advances in Financial Machine Learning* (2018), chapters 2–4.

# Outlines
1. Load prices
2. The triple-barrier method
3. Daily volatility: the barrier ruler
4. Three example trades
5. Label every stock
    - 5.1 Brute-force check
    - 5.2 Label distribution
6. Sensitivity to the settings
7. Triple-barrier vs fixed-horizon labels
8. Overlapping labels and sample uniqueness
9. CUSUM filter: sampling events instead of every day
10. Summary

In [ ]:
# ---------------------------------------------------------------- Imports
import sys
from pathlib import Path


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TICKERS, VOL_SPAN, PT_MULT, SL_MULT, HORIZON, CLASSES, LABEL_COLORS, PRICE_COLOR
from src.data import download_prices, save_processed
from src.labeling import daily_volatility, triple_barrier, label_all, average_uniqueness, cusum_filter
from src.plotting import set_style, savefig

# ---------------------------------------------------------------- Global constants
EXAMPLE = "AAPL"
MULT_GRID = [0.5, 0.75, 1.0, 1.5]          # barrier multipliers tested in the sensitivity analysis
HORIZON_GRID = [5, 10, 20]                 # vertical barriers tested (trading days)
N_CHECK = 300                              # random events verified by brute force
CHECK_SEED = 7
CUSUM_VOL_MULT = 1.0                       # CUSUM threshold = this x daily volatility

set_style()

# [1/10] Load prices
> **Objective:** <u>load</u> the cached prices from notebook 01 (the 14 stocks; `SPY` is only used as a feature later).

In [ ]:
prices = download_prices()
stocks = prices[prices.ticker.isin(TICKERS)].copy()
close = stocks.pivot(index="date", columns="ticker", values="close")

In [ ]:
print(f"{stocks.ticker.nunique()} stocks, {len(stocks):,} stock-days")

# [2/10] The triple-barrier method
> For an event (a potential trade) starting at day $t_0$ with closing price $P_0$ and daily volatility $\sigma_{t_0}$:
>
> | Barrier | Level | Label if touched first |
> |---|---|---|
> | upper (take-profit) | $P_0 \cdot e^{+m \,\sigma_{t_0} \sqrt{H}}$ | $+1$ |
> | lower (stop-loss) | $P_0 \cdot e^{-m \,\sigma_{t_0} \sqrt{H}}$ | $-1$ |
> | vertical (time limit) | $t_0 + H$ trading days | $0$ |
>
> - $m$ = barrier multiplier (`PT_MULT` = `SL_MULT` = 0.75), $H$ = horizon (`HORIZON` = 10 days).
> - $\sigma\sqrt{H}$ is the typical size of an $H$-day move (volatility grows with the square root of time), so the barriers mean
>   the same thing for a calm stock and a volatile one.
> - The label is known at $t_1$ = the day a barrier is touched (at most $t_0 + H$); $t_1$ matters for splitting and for overlaps.

# [3/10] Daily volatility: the barrier ruler
> **Formula:** exponentially weighted standard deviation of daily log returns,
> $\sigma_t = \sqrt{\text{EWMA}_{\text{span}=20}\left[(r_t - \bar r_t)^2\right]}$ — recent days weigh more, and it uses data up to $t$ only.

In [ ]:
vol = close.apply(lambda s: daily_volatility(s.dropna(), VOL_SPAN))
vol_summary = (vol * 100).describe().T[["mean", "min", "50%", "max"]].rename(columns={"50%": "median"})

fig, axes = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True)
axes[0].plot(close.index, close[EXAMPLE], color=PRICE_COLOR)
axes[0].set(yscale="log", title=f"{EXAMPLE} close (log scale)")
axes[1].plot(vol.index, vol[EXAMPLE] * 100, color="#dc2626")
axes[1].set(ylabel="daily volatility (%)", title=f"{EXAMPLE} EWM daily volatility (span {VOL_SPAN})")
savefig(fig, "02_volatility")
plt.close(fig)

In [ ]:
vol_summary.round(2)

In [ ]:
fig

**Answer:** volatility moves a lot over time (spikes in 2020 and during sell-offs) and differs between stocks, so the barrier width
changes with both: wide in turbulent periods, narrow in calm ones.

# [4/10] Three example trades
> **Objective:** <u>see</u> the barriers on real price paths — one example of each label.

In [ ]:
c_ex, v_ex = close[EXAMPLE].dropna(), vol[EXAMPLE].dropna()
lab_ex = triple_barrier(c_ex, v_ex.reindex(c_ex.index), HORIZON, PT_MULT, SL_MULT).dropna(subset=["label"])
examples = {k: lab_ex[(lab_ex.label == k) & (lab_ex.index.year == 2023)].index[5] for k in (1, -1, 0)}

fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for ax, (k, t0) in zip(axes, examples.items()):
    i = c_ex.index.get_loc(t0)
    window = c_ex.iloc[i - 5: i + HORIZON + 4]
    w = lab_ex.loc[t0, "barrier"]
    p0 = c_ex.iloc[i]
    ax.plot(window.index, window.values, color=PRICE_COLOR, marker="o", ms=3)
    ax.hlines([p0 * np.exp(PT_MULT * w), p0 * np.exp(-SL_MULT * w)], t0, c_ex.index[i + HORIZON],
              colors=[LABEL_COLORS[1], LABEL_COLORS[-1]], lw=2)
    ax.axvline(c_ex.index[i + HORIZON], color="black", ls="--", lw=1)
    ax.axvline(t0, color="black", lw=0.8)
    ax.scatter([lab_ex.loc[t0, "t1"]], [c_ex.loc[lab_ex.loc[t0, "t1"]]], s=120, color=LABEL_COLORS[k], zorder=5)
    ax.set_title(f"{CLASSES[k]} (label {k:+d}) — start {t0.date()}")
    ax.tick_params(axis="x", rotation=30)
savefig(fig, "02_examples")
plt.close(fig)

In [ ]:
lab_ex.loc[list(examples.values()), ["label", "ret", "barrier", "t1", "days_held"]].round(4)

In [ ]:
fig

Green and red lines are the take-profit and stop-loss barriers, the dashed line is the vertical barrier, and the large dot marks
where the label was decided ($t_1$).

# [5/10] Label every stock
> **Objective:** <u>label</u> every stock-day (except the last $H$ days of data, whose future is not complete yet).

In [ ]:
labels = label_all(stocks, VOL_SPAN, HORIZON, PT_MULT, SL_MULT)

In [ ]:
print(f"{len(labels):,} labelled events")
labels.head()

## 5.1 Brute-force check
> **Question:** does the fast labeling code give the same answer as a slow, obviously-correct loop?

In [ ]:
rng = np.random.default_rng(CHECK_SEED)
mismatches = 0
for k in rng.choice(len(labels), N_CHECK, replace=False):
    e = labels.iloc[k]
    c = close[e.ticker].dropna()
    i = c.index.get_loc(e.date)
    path = np.log(c.iloc[i + 1: i + 1 + HORIZON] / c.iloc[i])
    hits = [(j, +1) for j, x in enumerate(path) if x >= PT_MULT * e.barrier] + \
           [(j, -1) for j, x in enumerate(path) if x <= -SL_MULT * e.barrier]
    expected = min(hits)[1] if hits else 0
    mismatches += expected != e.label

In [ ]:
print(f"{N_CHECK} random events checked -> {mismatches} mismatches")

## 5.2 Label distribution
> **Question:** how balanced are the three classes, overall and per stock?

In [ ]:
overall = labels.label.map(CLASSES).value_counts(normalize=True).mul(100)
per_ticker = pd.crosstab(labels.ticker, labels.label.map(CLASSES), normalize="index").mul(100)
by_year = pd.crosstab(labels.date.dt.year, labels.label.map(CLASSES), normalize="index").mul(100)

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
per_ticker[list(CLASSES.values())].plot.barh(stacked=True, ax=axes[0], color=[LABEL_COLORS[k] for k in CLASSES], width=0.8)
axes[0].set(title="Label share per stock (%)", xlabel="%")
axes[0].legend(loc="lower right", fontsize=8)
by_year[list(CLASSES.values())].plot.bar(stacked=True, ax=axes[1], color=[LABEL_COLORS[k] for k in CLASSES], width=0.85)
axes[1].set(title="Label share per year (%)", ylabel="%")
axes[1].legend().remove()
savefig(fig, "02_label_distribution")
plt.close(fig)

In [ ]:
overall.round(1)

In [ ]:
fig

**Answer:** the classes are reasonably balanced overall, so plain accuracy is not misleading here, but macro-F1 and balanced
accuracy are still reported. Take-profit is more common than stop-loss for every one of the 14 stocks, even `INTC` and `PYPL` — they all
went up over the period (selection bias again) — and the mix changes from year to year with the market.

# [6/10] Sensitivity to the settings
> **Question:** how do the barrier multiplier $m$ and the horizon $H$ change the labels?

In [ ]:
rows = []
for h in HORIZON_GRID:
    for m in MULT_GRID:
        lab = label_all(stocks, VOL_SPAN, h, m, m)
        share = lab.label.value_counts(normalize=True)
        rows.append({"horizon": h, "mult": m, **{CLASSES[k]: share.get(k, 0) * 100 for k in CLASSES},
                     "mean_days_held": lab.days_held.mean()})
sensitivity = pd.DataFrame(rows)

fig, axes = plt.subplots(1, len(HORIZON_GRID), figsize=(17, 4.2), sharey=True)
for ax, h in zip(axes, HORIZON_GRID):
    s = sensitivity[sensitivity.horizon == h].set_index("mult")[list(CLASSES.values())]
    s.plot.bar(stacked=True, ax=ax, color=[LABEL_COLORS[k] for k in CLASSES], width=0.8, legend=False)
    ax.set(title=f"horizon {h} days", xlabel="barrier multiplier m", ylabel="%")
axes[-1].legend(loc="lower right", fontsize=8)
savefig(fig, "02_sensitivity")
plt.close(fig)

In [ ]:
sensitivity.round({"stop-loss": 1, "time-out": 1, "take-profit": 1, "mean_days_held": 1})

In [ ]:
fig

**Answer:** a wider barrier (larger $m$) turns more events into time-outs (from 14 % at $m$ = 0.5 to 77 % at $m$ = 1.5 for $H$ = 10).
Because the width scales with $\sqrt{H}$, the mix changes only moderately with the horizon (longer horizons give slightly fewer
time-outs). $m = 0.75$ with $H = 10$ gives 29 / 35 / 36 % and trades that last about 7 trading days on average — a balanced choice.

# [7/10] Triple-barrier vs fixed-horizon labels
> **Question:** how often does the simple label "price higher after $H$ days" disagree with what a stop-loss would have done?

In [ ]:
stocks = stocks.sort_values(["ticker", "date"])
stocks["fixed_sign"] = stocks.groupby("ticker")["close"].transform(lambda c: np.sign(np.log(c.shift(-HORIZON) / c)))
compare = labels.merge(stocks[["date", "ticker", "fixed_sign"]], on=["date", "ticker"])
crosstab = pd.crosstab(compare.label.map(CLASSES), compare.fixed_sign.map({-1: "lower after H", 0: "equal", 1: "higher after H"}),
                       normalize="index").mul(100)
stopped_but_up = ((compare.label == -1) & (compare.fixed_sign == 1)).mean() * 100

In [ ]:
print(f"events stopped out first but higher after {HORIZON} days: {stopped_but_up:.1f} % of all events")
crosstab.round(1)

**Answer:** 13 % of the stop-loss trades **still** ended higher after 10 days (3.7 % of all events), and 9 % of take-profit trades
ended lower. A fixed-horizon label gets these wrong from a trader's point of view: with a stop-loss, the first group lost money.
Time-outs split 57 / 42 between higher and lower, i.e. a fixed-horizon label turns small, noisy moves into confident up/down labels. Triple-barrier labels describe what actually happens to a trade.

# [8/10] Overlapping labels and sample uniqueness
> **Problem:** an event starting today and one starting tomorrow share up to 9 of the same future days, so their labels are
> **not independent**. Treating them as independent samples overstates how much information we have and inflates cross-validation.
>
> **Formulas** (per stock):
> - concurrency $c_t$ = number of events whose holding period $(t_0, t_1]$ contains day $t$
> - uniqueness of event $i$: $\bar u_i = \frac{1}{|T_i|}\sum_{t \in T_i} \frac{1}{c_t}$ — 1 = no overlap, $\frac{1}{H}$ = completely overlapped
> - used as **sample weights** in training, so overlapping events share their weight instead of counting many times

In [ ]:
u_check = average_uniqueness(lab_ex.index, lab_ex["t1"], c_ex.index)
uniq_summary = labels.groupby("ticker").uniqueness.mean().sort_values()
effective_n = labels.uniqueness.sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
sns.histplot(labels.uniqueness, bins=60, color=PRICE_COLOR, ax=axes[0])
axes[0].set(title="Average uniqueness of events", xlabel="uniqueness")
sns.boxplot(data=labels, x=labels.label.map(CLASSES), y="uniqueness", order=list(CLASSES.values()),
            palette={v: LABEL_COLORS[k] for k, v in CLASSES.items()}, hue=labels.label.map(CLASSES), legend=False, ax=axes[1])
axes[1].set(title="Uniqueness by label", xlabel="")
savefig(fig, "02_uniqueness")
plt.close(fig)

In [ ]:
print(f"events: {len(labels):,}  |  effective number of independent events (sum of uniqueness): {effective_n:,.0f}")
labels.groupby(labels.label.map(CLASSES)).uniqueness.mean().round(3)

In [ ]:
fig

**Answer:** the average event is only about 15 % unique: the 53,592 labelled events hold the information of roughly 8,300
independent trades. Stop-loss and take-profit events end early and overlap less, so they get more weight than
time-outs. → Models are trained with **uniqueness weights**, and validation uses **purging** (notebook 03).

# [9/10] CUSUM filter: sampling events instead of every day
> **Idea:** instead of a trade every day, start an event only after a meaningful move. The symmetric CUSUM filter accumulates
> returns and fires when the sum since the last event passes a threshold $h_t$:
> $S^+_t = \max(0, S^+_{t-1} + r_t)$, $S^-_t = \min(0, S^-_{t-1} + r_t)$; event when $S^+_t > h_t$ or $S^-_t < -h_t$ (then reset),
> with $h_t = \sigma_t$ here.

In [ ]:
events = cusum_filter(c_ex, v_ex * CUSUM_VOL_MULT)
cusum_labels = lab_ex.loc[lab_ex.index.intersection(events)]
cusum_compare = pd.DataFrame({
    "every day": lab_ex.label.map(CLASSES).value_counts(normalize=True) * 100,
    "CUSUM events": cusum_labels.label.map(CLASSES).value_counts(normalize=True) * 100,
}).loc[list(CLASSES.values())]
cusum_uniq = average_uniqueness(cusum_labels.index, cusum_labels["t1"], c_ex.index)

window = c_ex.loc["2023-01-01":"2023-06-30"]
fig, ax = plt.subplots(figsize=(14, 4.2))
ax.plot(window.index, window.values, color=PRICE_COLOR)
ev = events[(events >= window.index[0]) & (events <= window.index[-1])]
ax.scatter(ev, c_ex.loc[ev], color="#dc2626", s=25, zorder=5, label="CUSUM event")
ax.set(title=f"{EXAMPLE}: CUSUM events, first half of 2023")
ax.legend()
savefig(fig, "02_cusum")
plt.close(fig)

In [ ]:
print(f"{EXAMPLE}: {len(lab_ex):,} daily events -> {len(cusum_labels):,} CUSUM events; "
      f"mean uniqueness {lab_ex.index.map(u_check).to_series().mean():.3f} -> {cusum_uniq.mean():.3f}")
cusum_compare.round(1)

In [ ]:
fig

**Answer:** for `AAPL`, CUSUM keeps 1,672 of 4,211 events, raises mean uniqueness from 0.16 to 0.39, and leaves the label mix
almost unchanged. For the deep-learning models this project keeps
**every day** (more training samples) and handles overlap with uniqueness weights instead; CUSUM is kept as an option.

# [10/10] Summary

In [ ]:
save_processed(labels, "labels")
labels[["label", "uniqueness", "days_held"]].describe().round(3)

| Choice | Value | Reason |
|---|---|---|
| Volatility | EWM std of daily log returns, span 20 | adapts to calm and turbulent periods |
| Barriers | $\pm 0.75\,\sigma\sqrt{10}$ | about one third per class |
| Vertical barrier | 10 trading days | two trading weeks |
| Events | every stock-day | more samples for the sequence models |
| Overlap | uniqueness sample weights + purging | labels share future days |

Labels are saved to `data/processed/labels.parquet`.

# Part [3/5] 03_features_and_split

Repo notebook → [notebooks/03_features_and_split.ipynb](https://github.com/Pisit-Janthawee/dl-multiclassification-stock-triple-barrier-rnn-lstm/blob/main/notebooks/03_features_and_split.ipynb)

# Stock Triple-Barrier Classification · 03 Features & Purged Split

# Aim, Objective, and Motivation
- **Aim:** <u>describe</u> each day with stationary, causal features that a model can learn from.
- **Objective:** build technical, volatility, volume and market features, a fractionally differenced price,
  check which features relate to the labels, and split the data by time **with purging**.
- **Motivation:** my first version fed raw (scaled) prices to the networks. Here every feature is a return, ratio or
  oscillator, computed from data up to day $t$ only, and no event's outcome may overlap the next data split.

# Outlines
1. Load prices & labels
2. Fractional differentiation
3. Feature catalogue
4. Build features
5. Which features relate to the label?
6. Purged time split
7. Save dataset
8. Summary

In [ ]:
# ---------------------------------------------------------------- Imports
import sys
from pathlib import Path


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from src.config import TICKERS, MARKET, VALID_START, TEST_START, CLASSES, LABEL_COLORS, PRICE_COLOR
from src.data import download_prices, load_processed, save_processed
from src.evaluate import assign_split
from src.features import frac_diff, min_d_stationary, ticker_features, FEATURES
from src.plotting import set_style, savefig

# ---------------------------------------------------------------- Global constants
EXAMPLE = "AAPL"
FFD_THRESHOLD = 1e-3                 # drop fractional-difference weights smaller than this
ADF_ALPHA = 0.05
SPLIT_COLORS = {"train": "#4b5563", "valid": "#9ca3af", "test": "#dc2626", "purged": "black"}

set_style()

# [1/8] Load prices & labels

In [ ]:
prices = download_prices()
labels = load_processed("labels")
market = prices[prices.ticker == MARKET].set_index("date")

In [ ]:
print(f"prices: {prices.shape} | labels: {labels.shape}")

# [2/8] Fractional differentiation
> **Problem:** prices are non-stationary (notebook 01), returns are stationary but throw away all memory of the price level
> (returns = first difference, $d = 1$). Fractional differentiation uses a **fractional** $d \in (0, 1)$:
>
> $\tilde{x}_t = \sum_{k=0}^{K} w_k\, x_{t-k}$, with $w_0 = 1$, $w_k = -w_{k-1}\,\frac{d - k + 1}{k}$, truncated when $|w_k| < 10^{-3}$
>
> **Question:** what is the smallest $d$ that makes each stock's log price stationary (ADF p < 0.05)?
> Chosen on the **training period only**, so the choice does not peek at the future.

In [ ]:
d_rows, d_tables = [], {}
for t in TICKERS:
    lp = np.log(prices[(prices.ticker == t) & (prices.date < VALID_START)].set_index("date")["close"])
    d, table = min_d_stationary(lp, threshold=FFD_THRESHOLD, alpha=ADF_ALPHA)
    d_tables[t] = table
    corr = table.loc[table.d == d, "corr_with_price"].iloc[0]
    d_rows.append({"ticker": t, "min_d": d, "corr_with_log_price": corr})
d_choice = pd.DataFrame(d_rows).set_index("ticker")

tab = d_tables[EXAMPLE]
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(tab.d, tab.adf_p, marker="o", color=PRICE_COLOR, label="ADF p-value")
ax1.axhline(ADF_ALPHA, color=PRICE_COLOR, ls="--", lw=1)
ax2 = ax1.twinx()
ax2.plot(tab.d, tab.corr_with_price, marker="s", color="#dc2626", label="correlation with log price")
ax2.grid(False)
ax1.set(xlabel="d", ylabel="ADF p-value", title=f"{EXAMPLE}: stationarity vs memory as d grows")
ax2.set_ylabel("correlation with log price")
fig.legend(loc="center right", bbox_to_anchor=(0.88, 0.5))
savefig(fig, "03_frac_diff")
plt.close(fig)

In [ ]:
d_choice.round(3)

In [ ]:
fig

**Answer:** 13 of the 14 stocks become stationary at $d$ between 0.30 and 0.45, far below $d = 1$, while the fractionally
differenced series keeps a 0.92–0.99 correlation with the original log price. `IBM` needs $d = 0$: its log price was already
stationary over the training years (it moved sideways). Returns ($d = 1$) would have thrown that memory away. Each stock uses its own minimal $d$
for the feature `ffd_close`.

# [3/8] Feature catalogue
All features at day $t$ use data up to and including $t$. $C$ = close, $r$ = daily log return.

| Group | Feature | Formula / meaning |
|---|---|---|
| Momentum | `ret_1`, `ret_5`, `ret_10`, `ret_21` | $\ln(C_t / C_{t-k})$ |
| Volatility | `vol_21` | std of $r$ over 21 days |
| | `vol_ratio_5_21` | short / long volatility (is volatility rising?) |
| | `atr_14` | Average True Range / $C_t$, with $TR = \max(H - L, |H - C_{t-1}|, |L - C_{t-1}|)$ |
| | `bb_width` | Bollinger band width $4\sigma_{20} / SMA_{20}$ |
| Oscillators | `rsi_14` | $1 - \frac{1}{1 + RS}$, $RS$ = average gain / average loss (Wilder, 14 days), scaled to [0, 1] |
| | `macd_hist` | $(EMA_{12} - EMA_{26}) - \text{signal}_9$, divided by $C_t$ |
| | `bb_pctb` | position inside the Bollinger bands: $\frac{C - (SMA_{20} - 2\sigma)}{4\sigma}$ |
| Trend | `dist_sma_50`, `dist_sma_200`, `dist_ema_21` | $C_t / MA - 1$ (from the SMA / EMA of my first version) |
| Volume | `volume_z` | z-score of log volume vs its 21-day mean |
| Memory | `ffd_close` | fractionally differenced log price (section 2) |
| Market | `mkt_ret_1`, `mkt_ret_5`, `mkt_vol_21` | the same for `SPY` |
| | `rel_strength_21` | stock 21-day return minus `SPY` 21-day return |
| | `beta_63` | $\frac{\text{Cov}(r, r_{SPY})}{\text{Var}(r_{SPY})}$ over 63 days |

# [4/8] Build features
> **Objective:** <u>compute</u> the features per stock and join them to the labels. The first ~200 days of each stock are dropped
> (the 200-day average needs a full window).

In [ ]:
parts = []
for t in TICKERS:
    g = prices[prices.ticker == t].set_index("date")
    f = ticker_features(g, market, d_choice.loc[t, "min_d"])
    f["ticker"] = t
    parts.append(f.reset_index())
features = pd.concat(parts, ignore_index=True)
dataset = labels.merge(features, on=["date", "ticker"], how="inner").dropna(subset=FEATURES).reset_index(drop=True)

In [ ]:
print(f"labelled events: {len(labels):,} -> with complete features: {len(dataset):,}")
dataset[FEATURES].describe().T[["mean", "std", "min", "max"]].round(3)

# [5/8] Which features relate to the label?
> **Question:** do feature values differ between take-profit, time-out and stop-loss events (training period only)?
>
> **Kruskal-Wallis H test** (non-parametric one-way ANOVA on ranks), $H_0$: the three groups have the same distribution.
> **Effect size:** $\eta^2_H = \frac{H - k + 1}{n - k}$ with $k = 3$ groups — 0.01 small, 0.06 medium, 0.14 large.
> With tens of thousands of rows small effects are "significant", so the effect size matters most.

In [ ]:
train_part = dataset[dataset.date < VALID_START]
rows = []
for f in FEATURES:
    groups = [train_part.loc[train_part.label == k, f] for k in CLASSES]
    h, p = stats.kruskal(*groups)
    rows.append({"feature": f, "H": h, "p_value": p, "eta2": (h - 3 + 1) / (len(train_part) - 3),
                 **{f"median_{CLASSES[k]}": g.median() for k, g in zip(CLASSES, groups)}})
kruskal = pd.DataFrame(rows).sort_values("eta2", ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
s = kruskal.set_index("feature")["eta2"].sort_values()
ax.barh(s.index, s.values, color=PRICE_COLOR)
ax.set(title="Kruskal-Wallis effect size per feature (train)", xlabel="eta squared")
savefig(fig, "03_feature_effects")
plt.close(fig)

In [ ]:
kruskal.round(4)

In [ ]:
fig

**Answer:** every effect is **small** (largest $\eta^2$ = 0.037 for `vol_21`, then 0.026 for `bb_width` and 0.011 for `atr_14`):
the strongest features are related to **volatility** (they mainly separate time-outs from
barrier touches, because the barrier width is itself volatility-based), while momentum and oscillator features barely differ
between take-profit and stop-loss. This matches notebook 01: volatility is somewhat predictable, direction much less.
Expect modest model performance — the honest baseline for a hard problem.

# [6/8] Purged time split
> **Objective:** <u>split</u> by event start date — train < 2022, valid 2022–2023, test 2024 onwards — and **purge** events whose
> label end $t_1$ crosses into the next set.
>
> **Why purge?** An event starting on 2021-12-28 with $t_1$ = 2022-01-07 is labelled with January 2022 prices, which belong to the
> validation period. Keeping it in training leaks validation information into the model.

In [ ]:
dataset["split"] = assign_split(dataset, VALID_START, TEST_START)
split_summary = dataset.groupby("split").agg(events=("label", "size"), start=("date", "min"), end=("date", "max"),
                                             take_profit=("label", lambda s: (s == 1).mean()),
                                             stop_loss=("label", lambda s: (s == -1).mean()),
                                             mean_uniqueness=("uniqueness", "mean"))
split_summary = split_summary.loc[["train", "purged", "valid", "test"]]

counts = dataset.groupby([dataset.date.dt.to_period("M"), "split"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 3.8))
bottom = np.zeros(len(counts))
for s_name, c in SPLIT_COLORS.items():
    if s_name in counts:
        ax.bar(counts.index.to_timestamp(), counts[s_name], bottom=bottom, width=25, color=c, label=s_name)
        bottom += counts[s_name].to_numpy()
ax.set(title="Events per month by split", ylabel="events")
ax.legend(loc="upper left")
savefig(fig, "03_split")
plt.close(fig)

In [ ]:
split_summary.round(3)

In [ ]:
fig

**Answer:** 244 events are purged (about 9 per stock at each of the two boundaries; most are stop-losses that had not finished
when the period ended). Without purging they would let the model see the first days of the next period.
The class mix shifts between periods (take-profit 37.5 % in train vs 34–35 % later), another reason to judge models with macro-F1.

# [7/8] Save dataset
> Features are saved **unscaled**; scaling is fitted on the training split inside notebook 04.

In [ ]:
keep = ["date", "ticker", "label", "t1", "ret", "barrier", "vol", "days_held", "uniqueness", "split"] + FEATURES
save_processed(dataset[keep], "dataset")
save_processed(d_choice.reset_index(), "frac_diff_d")

In [ ]:
print(f"saved {len(dataset):,} rows x {len(keep)} columns")
dataset.split.value_counts()

# [8/8] Summary
- **21 features**: momentum, volatility, oscillators, trend distance, volume, a fractionally differenced price and market context.
- Fractional differentiation makes prices stationary at $d$ = 0.30–0.45 (IBM: 0) while keeping memory that returns lose.
- Feature–label relationships are **weak** (all small effect sizes) — volatility features matter most.
- **Purged time split:** train < 2022, valid 2022–2023, test 2024 onwards.

# Part [4/5] 04_modeling

Repo notebook → [notebooks/04_modeling.ipynb](https://github.com/Pisit-Janthawee/dl-multiclassification-stock-triple-barrier-rnn-lstm/blob/main/notebooks/04_modeling.ipynb)

# Stock Triple-Barrier Classification · 04 Modeling

# Aim, Objective, and Motivation
- **Aim:** <u>find out</u> how well the triple-barrier outcome can be predicted, and whether sequence models
  (SimpleRNN, LSTM, GRU — the models of my first version) beat simpler ones.
- **Objective:** compare a prior baseline, logistic regression, LightGBM and three recurrent networks on the **validation** period
  (2022–2023), with uniqueness sample weights, and show how much purging changes cross-validation scores.
- **Motivation:** for a hard, noisy target the important question is not "which model is best" but "is anything better than guessing".

# Outlines
1. Load dataset
2. Metrics
3. Scaling & sample weights
4. Baselines: prior & logistic regression
5. LightGBM
6. Purged vs naive cross-validation
7. Sequences for recurrent networks
8. SimpleRNN, LSTM, GRU
9. Benchmark & save predictions
10. Summary

In [ ]:
# ---------------------------------------------------------------- Imports
import os
os.environ["KERAS_BACKEND"] = "torch"          # Keras 3 on PyTorch -> uses the GPU on Windows

import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import keras
import lightgbm as lgb
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from src.config import CLASSES, LABEL_COLORS, MODELS_DIR, PRICE_COLOR, RANDOM_STATE, REPORTS_DIR
from src.data import load_processed, save_processed
from src.evaluate import LABELS, PurgedKFold, make_sequences, score
from src.features import FEATURES
from src.plotting import set_style, savefig

# ---------------------------------------------------------------- Global constants
SEED = RANDOM_STATE
LOOKBACK = 30                      # days of features in each sequence
EPOCHS = 60
BATCH_SIZE = 256
PATIENCE = 8                       # early-stopping patience (epochs)
UNITS = 64                         # recurrent units
DROPOUT = 0.3
LEARNING_RATE = 1e-3
CLIP = 5.0                         # clip standardised features to +-5 to tame extreme days
CV_FOLDS = 5
EMBARGO_DAYS = 10
TO_CLASS = {-1: 0, 0: 1, 1: 2}     # label -> class index (column order of every probability matrix)
LGB_PARAMS = dict(objective="multiclass", n_estimators=2000, learning_rate=0.02, num_leaves=15, min_child_samples=200,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, verbose=-1)

set_style()
keras.utils.set_random_seed(SEED)
DEVICE = "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

# [1/10] Load dataset
> **Objective:** <u>load</u> the features and labels from notebook 03; purged events are not used.

In [ ]:
data = load_processed("dataset").sort_values(["ticker", "date"]).reset_index(drop=True)
data["y"] = data["label"].map(TO_CLASS)
is_train, is_valid, is_test = (data.split == s for s in ("train", "valid", "test"))

In [ ]:
print(f"Keras {keras.__version__} on {keras.backend.backend()} ({DEVICE})")
data.split.value_counts()

# [2/10] Metrics
> **Question:** how do we judge a 3-class model on a noisy financial target?
>
> - **Macro-F1** $= \frac{1}{3}\sum_{c} F1_c$ — every class counts equally; a model predicting one class scores at most ~0.17.
> - **Balanced accuracy** $= \frac{1}{3}\sum_c \text{recall}_c$ — 0.333 is random guessing.
> - **Log-loss** $= -\frac{1}{n}\sum_i \sum_c y_{ic} \ln \hat p_{ic}$ (uniqueness-weighted) — rewards honest probabilities;
>   the **prior** baseline (always predict the training class frequencies) sets the bar.
> - **Take-profit precision** — of the days predicted "take-profit", how many really were? This is what a long-only strategy needs.

# [3/10] Scaling & sample weights
> - **Standardisation** $z = \frac{x - \mu_{train}}{\sigma_{train}}$, fitted on **training rows only**, then clipped to $\pm 5$.
> - **Sample weight** = event uniqueness (notebook 02), rescaled to mean 1 in each split, so overlapping events share weight.

In [ ]:
scaler = StandardScaler().fit(data.loc[is_train, FEATURES])
data[FEATURES] = np.clip(scaler.transform(data[FEATURES]), -CLIP, CLIP).astype(np.float32)
data["w"] = data["uniqueness"] / data.groupby("split")["uniqueness"].transform("mean")

X = {s: data.loc[m, FEATURES] for s, m in [("train", is_train), ("valid", is_valid), ("test", is_test)]}
y = {s: data.loc[m, "y"].to_numpy() for s, m in [("train", is_train), ("valid", is_valid), ("test", is_test)]}
w = {s: data.loc[m, "w"].to_numpy() for s, m in [("train", is_train), ("valid", is_valid), ("test", is_test)]}
y_label = {s: np.array(LABELS)[v] for s, v in y.items()}

In [ ]:
pd.DataFrame({s: [len(y[s]), w[s].mean()] for s in y}, index=["rows", "mean weight"]).round(3)

Helper: every model's validation and test probabilities are stored for the benchmark and for notebook 05.

In [ ]:
results, probas = [], {"valid": {}, "test": {}}


def record(name, p_valid, p_test):
    probas["valid"][name], probas["test"][name] = p_valid, p_test
    results.append(score(y_label["valid"], p_valid, name, weights=w["valid"]))


def results_table():
    return pd.DataFrame(results).drop_duplicates("model", keep="last").set_index("model").round(4)

# [4/10] Baselines: prior & logistic regression
> - **Prior:** predict the training class frequencies for every day (no information at all).
> - **Multinomial logistic regression:** $P(y = c \mid x) = \frac{e^{w_c^\top x}}{\sum_k e^{w_k^\top x}}$ — a linear model.

In [ ]:
prior = np.bincount(y["train"], weights=w["train"], minlength=3) / w["train"].sum()
record("Prior (class frequencies)", np.tile(prior, (len(y["valid"]), 1)), np.tile(prior, (len(y["test"]), 1)))

logreg = LogisticRegression(max_iter=2000, C=0.1).fit(X["train"], y["train"], sample_weight=w["train"])
record("Logistic Regression", logreg.predict_proba(X["valid"]), logreg.predict_proba(X["test"]))

In [ ]:
results_table()

# [5/10] LightGBM
> Gradient-boosted trees on the same 21 features of the event day (no sequence). Early stopping on the validation log-loss
> chooses the number of trees.

In [ ]:
lgbm = lgb.LGBMClassifier(**LGB_PARAMS)
lgbm.fit(X["train"], y["train"], sample_weight=w["train"],
         eval_set=[(X["valid"], y["valid"])], eval_sample_weight=[w["valid"]],
         callbacks=[lgb.early_stopping(100, verbose=False)])
record("LightGBM", lgbm.predict_proba(X["valid"]), lgbm.predict_proba(X["test"]))
joblib.dump(lgbm, MODELS_DIR / "lightgbm.joblib")

In [ ]:
print(f"best iteration: {lgbm.best_iteration_}")
results_table()

# [6/10] Purged vs naive cross-validation
> **Question:** how much does ordinary K-fold cross-validation overstate performance when labels overlap in time?
>
> - **Naive K-fold (shuffled):** neighbouring days — which share most of their future prices — land in train and test folds.
> - **Purged K-fold with embargo:** folds are contiguous blocks of time; training events overlapping the test block are removed
>   (purging) and so are events starting within 10 days after it (embargo).

In [ ]:
cv_data = data.loc[is_train].reset_index(drop=True)
cv_model = dict(LGB_PARAMS, n_estimators=300)
cv_rows = []
splitters = {"naive shuffled K-fold": KFold(CV_FOLDS, shuffle=True, random_state=SEED).split(cv_data),
             "purged K-fold + embargo": PurgedKFold(CV_FOLDS, EMBARGO_DAYS).split(cv_data["date"], cv_data["t1"])}
for cv_name, folds in splitters.items():
    for k, (tr, te) in enumerate(folds):
        m = lgb.LGBMClassifier(**cv_model).fit(cv_data.loc[tr, FEATURES], cv_data.loc[tr, "y"],
                                               sample_weight=cv_data.loc[tr, "w"])
        s = score(np.array(LABELS)[cv_data.loc[te, "y"]], m.predict_proba(cv_data.loc[te, FEATURES]), cv_name,
                  weights=cv_data.loc[te, "w"])
        cv_rows.append({**s, "fold": k, "train_rows": len(tr)})
cv_results = pd.DataFrame(cv_rows)
cv_summary = cv_results.groupby("model")[["macro_f1", "balanced_acc", "log_loss", "train_rows"]].mean()

In [ ]:
cv_summary.round(4)

**Answer:** shuffled K-fold reports macro-F1 **0.472** and log-loss 0.990; purged K-fold with embargo reports **0.394** and 1.068.
The gap comes from overlapping neighbours of each test day sitting in the training folds. On the truly unseen validation period
LightGBM reaches 0.333 macro-F1 and log-loss 1.084, so purged CV is far closer to reality, though still somewhat optimistic
(the market regime also changes between periods). Shuffled K-fold should never be used for this kind of data.

# [7/10] Sequences for recurrent networks
> **Objective:** <u>give</u> each event the last 30 days of features of its stock: input shape $(n, 30, 21)$.
> Windows may reach back into an earlier split — that is past information, available at prediction time.

In [ ]:
X_seq_all, row_idx = make_sequences(data, FEATURES, LOOKBACK)
pos = pd.Series(np.arange(len(row_idx)), index=row_idx)
seq = {}
for s, m in [("train", is_train), ("valid", is_valid), ("test", is_test)]:
    rows = data.index[m]
    keep = rows[np.isin(rows, row_idx)]
    seq[s] = (X_seq_all[pos[keep].to_numpy()], data.loc[keep, "y"].to_numpy(), data.loc[keep, "w"].to_numpy(), keep)

In [ ]:
{s: v[0].shape for s, v in seq.items()}

The sequence sets are slightly smaller (each stock's first 29 events have no full window), so for a fair comparison the
non-sequence models are also scored on exactly these rows in the benchmark.

# [8/10] SimpleRNN, LSTM, GRU
> All three use the same architecture: recurrent layer (64 units) → dropout 0.3 → dense 32 (ReLU) → softmax over 3 classes,
> trained with Adam on the uniqueness-weighted cross-entropy, early stopping on validation loss.
>
> - **SimpleRNN:** $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$ — suffers from vanishing gradients on long sequences.
> - **LSTM:** gates $f_t, i_t, o_t = \sigma(\cdot)$ control a cell state $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$, $h_t = o_t \odot \tanh(c_t)$.
> - **GRU:** update gate $z_t$ and reset gate $r_t$; $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t$ — fewer parameters than LSTM.

In [ ]:
LAYERS = {"SimpleRNN": keras.layers.SimpleRNN, "LSTM": keras.layers.LSTM, "GRU": keras.layers.GRU}
histories, fit_seconds = {}, {}
Xtr, ytr, wtr, _ = seq["train"]
Xva, yva, wva, _ = seq["valid"]
Xte = seq["test"][0]
for name, layer in LAYERS.items():
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential([
        keras.Input((LOOKBACK, len(FEATURES))),
        layer(UNITS),
        keras.layers.Dropout(DROPOUT),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(3, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss="sparse_categorical_crossentropy",
                  weighted_metrics=["accuracy"])
    start = time.time()
    hist = model.fit(Xtr, ytr, sample_weight=wtr, validation_data=(Xva, yva, wva), epochs=EPOCHS, batch_size=BATCH_SIZE,
                     callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)], verbose=0)
    fit_seconds[name] = time.time() - start
    histories[name] = pd.DataFrame(hist.history)
    model.save(MODELS_DIR / f"{name.lower()}.keras")
    record(name, model.predict(Xva, verbose=0), model.predict(Xte, verbose=0))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.2), sharey=True)
for ax, (name, h) in zip(axes, histories.items()):
    ax.plot(h["loss"], color=PRICE_COLOR, label="train")
    ax.plot(h["val_loss"], color="#dc2626", label="validation")
    ax.axvline(h["val_loss"].idxmin(), color="black", ls=":", lw=1)
    ax.set(title=f"{name}: loss per epoch", xlabel="epoch")
    ax.legend()
savefig(fig, "04_training_curves")
plt.close(fig)

In [ ]:
pd.DataFrame({"epochs_run": {k: len(h) for k, h in histories.items()},
              "best_epoch": {k: int(h["val_loss"].idxmin()) + 1 for k, h in histories.items()},
              "fit_seconds": fit_seconds}).round(1)

In [ ]:
fig

**Answer:** validation loss is lowest at epoch 7 (SimpleRNN) and epoch 4 (LSTM, GRU), then rises while training loss keeps falling:
the networks start memorising noise almost immediately. Early stopping restores the best epoch; each network trains in 2–3 minutes
on the GPU.

# [9/10] Benchmark & save predictions
> **Objective:** <u>compare</u> all models on the **same** validation rows (those with a full 30-day window).

In [ ]:
valid_rows = seq["valid"][3]
test_rows = seq["test"][3]
v_pos = pd.Series(np.arange(is_valid.sum()), index=data.index[is_valid])
t_pos = pd.Series(np.arange(is_test.sum()), index=data.index[is_test])
yv = np.array(LABELS)[data.loc[valid_rows, "y"]]
wv = data.loc[valid_rows, "w"].to_numpy()
bench = []
for name, p in probas["valid"].items():
    p_same = p if name in LAYERS else p[v_pos[valid_rows].to_numpy()]
    bench.append(score(yv, p_same, name, weights=wv))
benchmark = pd.DataFrame(bench).set_index("model").sort_values("log_loss")

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
b = benchmark.sort_values("macro_f1")
axes[0].barh(b.index, b["macro_f1"], color=PRICE_COLOR)
axes[0].axvline(1 / 3, color="black", ls=":", lw=1)
axes[0].set(title="Validation macro-F1", xlim=(0, max(0.5, b["macro_f1"].max() * 1.15)))
b = benchmark.sort_values("log_loss", ascending=False)
axes[1].barh(b.index, b["log_loss"], color=["#dc2626" if i.startswith("Prior") else PRICE_COLOR for i in b.index])
axes[1].set(title="Validation log-loss (lower is better; red = prior baseline)",
            xlim=(b["log_loss"].min() * 0.97, b["log_loss"].max() * 1.02))
savefig(fig, "04_benchmark")
plt.close(fig)

# save validation + test probabilities of every model (same rows for all) for notebook 05
out = []
for split_name, rows, posmap in [("valid", valid_rows, v_pos), ("test", test_rows, t_pos)]:
    base = data.loc[rows, ["date", "ticker", "label", "t1", "ret", "uniqueness"]].reset_index(drop=True)
    for name, p in probas[split_name].items():
        p_same = p if name in LAYERS else p[posmap[rows].to_numpy()]
        for c, lab in enumerate(LABELS):
            base[f"{name}|p{lab:+d}"] = p_same[:, c]
    base["split"] = split_name
    out.append(base)
save_processed(pd.concat(out, ignore_index=True), "predictions")
benchmark.to_csv(REPORTS_DIR / "validation_benchmark.csv")
json.dump({"features": FEATURES, "lookback": LOOKBACK, "scaler_mean": scaler.mean_.tolist(),
           "scaler_scale": scaler.scale_.tolist(), "clip": CLIP}, open(REPORTS_DIR / "preprocessing.json", "w"), indent=2)

In [ ]:
benchmark

In [ ]:
fig

**Answer:** every model beats the prior baseline, but only by a little.
- **SimpleRNN** has the best log-loss (1.080 vs 1.100 for the prior) and balanced accuracy (0.414 vs 0.333), with take-profit
  precision 0.41 against a base rate of 0.35.
- LightGBM (1.084) and GRU (1.087) follow; the LSTM is the weakest learned model on log-loss (1.094).
- The recurrent networks reach a higher macro-F1 (0.36–0.37 vs 0.33) partly because they predict stop-loss more often
  (SimpleRNN 5 %, GRU 7 %, LSTM 18 % of validation days vs 2–3 % for logistic regression and LightGBM). All models mostly
  predict take-profit (51–59 %) or time-out.

The differences between models are small compared with the gap to the prior, and come from a single two-year period.
Notebook 05 checks whether the edge survives on the test period and in a backtest with costs.

# [10/10] Summary
- Six models compared on the same validation rows, with uniqueness weights and a purged split.
- All learned models beat the prior baseline by a small margin; SimpleRNN is best on log-loss (1.080 vs 1.100).
- Purged cross-validation (macro-F1 0.394) is much more honest than shuffled K-fold (0.472).
- Validation and test probabilities of every model are saved to `data/processed/predictions.parquet` for notebook 05.

# Part [5/5] 05_evaluation_backtest

Repo notebook → [notebooks/05_evaluation_backtest.ipynb](https://github.com/Pisit-Janthawee/dl-multiclassification-stock-triple-barrier-rnn-lstm/blob/main/notebooks/05_evaluation_backtest.ipynb)

# Stock Triple-Barrier Classification · 05 Evaluation & Backtest

# Aim, Objective, and Motivation
- **Aim:** <u>test</u> whether the small validation edge survives on unseen data (2024-01 to 2026-10) and whether it is worth
  anything as a trading rule after costs.
- **Objective:** score every model once on the test period, pick a trading threshold on validation, backtest a long-only rule
  against buy-and-hold and random entries, and explain the LightGBM model with SHAP.
- **Motivation:** classification metrics a little above chance can still be worthless (or useful) once costs, overlapping trades and
  the market's own trend are included.

# Outlines
1. Load predictions & prices
2. Test-period classification metrics
3. Confusion matrix of the best validation model
4. Are the probabilities meaningful? (take-profit reliability)
5. Trading rule & threshold (validation)
6. Backtest on the test period
7. Is it better than luck? Random-entry benchmark
8. Explainability: SHAP for LightGBM
9. Summary & limitations

In [ ]:
# ---------------------------------------------------------------- Imports
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.metrics import confusion_matrix

from src.config import TICKERS, CLASSES, LABEL_COLORS, PRICE_COLOR, MODELS_DIR, REPORTS_DIR, RANDOM_STATE, TEST_START
from src.data import download_prices, load_processed
from src.evaluate import LABELS, score, backtest_long, performance
from src.features import FEATURES
from src.plotting import set_style, savefig

# ---------------------------------------------------------------- Global constants
MODELS = ["Logistic Regression", "LightGBM", "SimpleRNN", "LSTM", "GRU"]
BEST_VALID_MODEL = "SimpleRNN"          # best validation log-loss in notebook 04
COST_BPS = 10                           # cost per side (commission + spread), basis points
THRESHOLD_GRID = np.round(np.arange(0.30, 0.66, 0.02), 2)   # P(take-profit) thresholds tried on validation
MIN_TRADES = 50                         # ignore thresholds giving fewer validation trades
N_RANDOM = 200                          # random-entry simulations
RELIABILITY_BINS = [0, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 1.0]
SHAP_SAMPLE = 3000
STRATEGY_COLORS = {"Buy & hold (same stocks)": "#9ca3af", "Random entries (median)": "black"}

set_style()

# [1/9] Load predictions & prices
> **Objective:** <u>load</u> the validation and test probabilities saved in notebook 04 (same rows for every model).

In [ ]:
pred = load_processed("predictions")
prices = download_prices()
stock_prices = prices[prices.ticker.isin(TICKERS)]
valid, test = pred[pred.split == "valid"].reset_index(drop=True), pred[pred.split == "test"].reset_index(drop=True)
P = {s: {m: d[[f"{m}|p{l:+d}" for l in LABELS]].to_numpy() for m in MODELS + ["Prior (class frequencies)"]}
     for s, d in [("valid", valid), ("test", test)]}

In [ ]:
print(f"valid: {len(valid):,} events ({valid.date.min().date()} to {valid.date.max().date()}) | "
      f"test: {len(test):,} events ({test.date.min().date()} to {test.date.max().date()})")

# [2/9] Test-period classification metrics
> **Question:** does any model still beat the prior baseline on 2024–2026 data it has never seen?

In [ ]:
w_test = test["uniqueness"] / test["uniqueness"].mean()
test_metrics = pd.DataFrame([score(test["label"].to_numpy(), P["test"][m], m, weights=w_test)
                             for m in ["Prior (class frequencies)"] + MODELS]).set_index("model")
test_metrics["log_loss_vs_prior"] = test_metrics["log_loss"] - test_metrics.loc["Prior (class frequencies)", "log_loss"]
valid_metrics = pd.read_csv(REPORTS_DIR / "validation_benchmark.csv", index_col=0)
compare = pd.DataFrame({"valid_log_loss": valid_metrics["log_loss"], "test_log_loss": test_metrics["log_loss"],
                        "valid_balanced_acc": valid_metrics["balanced_acc"], "test_balanced_acc": test_metrics["balanced_acc"]})

In [ ]:
test_metrics.sort_values("log_loss").round(4)

In [ ]:
compare.sort_values("test_log_loss").round(4)

**Answer:** yes, by a small margin. Every model beats the prior on test log-loss (SimpleRNN 1.060 vs 1.099, GRU 1.064,
logistic regression 1.067, LightGBM 1.071, LSTM 1.072) and reaches balanced accuracy 0.40–0.41 vs 0.33 for guessing.
The test edge is similar to validation, so it is not an artefact of model selection. SimpleRNN, chosen on validation,
is also best on test log-loss, but the gaps between models (≤ 0.012) are much smaller than the gap to the prior.

# [3/9] Confusion matrix of the best validation model
> **Objective:** <u>see</u> which outcomes the model confuses. Rows = true label, columns = predicted label (row-normalised).

In [ ]:
pred_best = np.array(LABELS)[P["test"][BEST_VALID_MODEL].argmax(1)]
cm = confusion_matrix(test["label"], pred_best, labels=LABELS, normalize="true")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm, cmap="Greys", vmin=0, vmax=1)
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v > 0.5 else "black", fontsize=13)
names = [CLASSES[k] for k in LABELS]
ax.set(xticks=range(3), yticks=range(3), xticklabels=names, yticklabels=names, xlabel="predicted", ylabel="true",
       title=f"{BEST_VALID_MODEL}: test confusion matrix")
ax.grid(False)
savefig(fig, "05_confusion")
plt.close(fig)

In [ ]:
pd.Series(pred_best).map(CLASSES).value_counts(normalize=True).round(3).rename("share of test predictions")

In [ ]:
fig

**Answer:** the model separates **"will the price move enough to hit a barrier?"** much better than **"which way?"**:
60 % of time-outs are recognised, but true stop-losses are predicted as take-profit 60 % of the time — about as often as
true take-profits (57 %). It predicts stop-loss for only 5.5 % of test days. This matches the features: volatility features
carry most of the signal (notebook 03) and direction is close to unpredictable (notebook 01).

# [4/9] Are the probabilities meaningful? (take-profit reliability)
> **Question:** when a model gives a higher P(take-profit), does take-profit really happen more often?
> A useful model shows an **increasing** observed rate across probability bins.

In [ ]:
rel_rows = []
for m in ["LightGBM", BEST_VALID_MODEL]:
    p_tp = P["test"][m][:, 2]
    g = pd.DataFrame({"p": p_tp, "tp": test["label"].eq(1)}).groupby(pd.cut(p_tp, RELIABILITY_BINS), observed=True)
    r = g.agg(events=("tp", "size"), mean_predicted=("p", "mean"), observed_tp_rate=("tp", "mean")).reset_index()
    r["model"] = m
    rel_rows.append(r)
reliability = pd.concat(rel_rows)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot([0.2, 0.7], [0.2, 0.7], ls="--", c="grey", label="perfect")
for m, c in [("LightGBM", PRICE_COLOR), (BEST_VALID_MODEL, "#dc2626")]:
    r = reliability[(reliability.model == m) & (reliability.events >= 30)]
    ax.plot(r.mean_predicted, r.observed_tp_rate, marker="o", color=c, label=m)
ax.axhline(test["label"].eq(1).mean(), color="black", lw=0.8, ls=":", label="test take-profit rate")
ax.set(xlabel="predicted P(take-profit)", ylabel="observed take-profit rate", title="Take-profit reliability (test)")
ax.legend()
savefig(fig, "05_reliability")
plt.close(fig)

In [ ]:
reliability.round(3)

In [ ]:
fig

**Answer:** partly. For both models the observed take-profit rate **rises** with the predicted probability (LightGBM: 24 % in the
lowest bin to 54 % at 0.50–0.55; SimpleRNN: 24 % to 60 % above 0.60), so the ranking carries information. SimpleRNN is
over-confident in the middle (predicted 0.52, observed 0.40), and LightGBM is slightly over-confident everywhere.

# [5/9] Trading rule & threshold (validation)
> **Rule (long-only):** on day $t_0$, if $P(\text{take-profit}) \ge \tau$ and the stock is not already held, buy at the close and sell
> at the close of $t_1$ (take-profit, stop-loss or day 10). One position per stock; capital split equally across the 14 stocks;
> idle capital earns 0. Cost: 10 bp on entry and on exit.
>
> **Metrics:** Sharpe $= \frac{\bar r}{\sigma_r}\sqrt{252}$ (daily returns, no risk-free rate), CAGR, maximum drawdown
> $= \min_t \left(\frac{E_t}{\max_{s \le t} E_s} - 1\right)$.
>
> $\tau$ is chosen per model to maximise the **validation** Sharpe (at least 50 trades), then frozen for the test period.

In [ ]:
def portfolio(events, signal, period_prices):
    daily, trades = backtest_long(events, signal, period_prices, COST_BPS)
    simple = (np.exp(daily) - 1).reindex(columns=TICKERS).fillna(0).mean(axis=1)   # equal capital per stock
    return np.log1p(simple), trades


def period_slice(events):
    start, end = events.date.min(), events.t1.max()
    return stock_prices[(stock_prices.date >= start - pd.Timedelta(days=5)) & (stock_prices.date <= end)], start, end


valid_prices, v_start, v_end = period_slice(valid)
tau_rows = []
for m in MODELS:
    p_tp = pd.Series(P["valid"][m][:, 2])
    for tau in THRESHOLD_GRID:
        sig = p_tp >= tau
        if sig.sum() < MIN_TRADES:
            continue
        r, trades = portfolio(valid, sig, valid_prices)
        r = r[(r.index > v_start) & (r.index <= v_end)]
        tau_rows.append({"model": m, "tau": tau, "signals": int(sig.sum()), "trades": len(trades), **performance(r)})
tau_table = pd.DataFrame(tau_rows)
best_tau = tau_table.loc[tau_table.groupby("model")["sharpe"].idxmax()].set_index("model")

In [ ]:
best_tau[["tau", "signals", "trades", "sharpe", "cagr", "max_drawdown"]].round(3)

**Answer:** the best validation thresholds lie between 0.46 and 0.58. Choosing $\tau$ by maximising Sharpe over a grid on one
two-year period is itself a form of overfitting, which the test period will reveal.

# [6/9] Backtest on the test period
> **Question:** with thresholds frozen from validation, does the rule beat buying and holding the same 14 stocks?

In [ ]:
test_prices, t_start, t_end = period_slice(test)
close = test_prices.pivot(index="date", columns="ticker", values="close")
bh = np.log1p((close.pct_change()).reindex(columns=TICKERS).mean(axis=1))      # equal-weight, daily rebalanced
bh = bh[(bh.index > t_start) & (bh.index <= t_end)]

curves, perf_rows, trade_stats = {"Buy & hold (same stocks)": bh}, [], []
perf_rows.append({"strategy": "Buy & hold (same stocks)", **performance(bh), "exposure": 1.0, "trades": np.nan})
for m in MODELS:
    sig = pd.Series(P["test"][m][:, 2] >= best_tau.loc[m, "tau"])
    r, trades = portfolio(test, sig, test_prices)
    r = r[(r.index > t_start) & (r.index <= t_end)]
    curves[m] = r
    exposure = (r != 0).mean()
    perf_rows.append({"strategy": m, **performance(r), "exposure": exposure, "trades": len(trades)})
    trade_stats.append({"model": m, "trades": len(trades), "take_profit_share": (trades.label == 1).mean(),
                        "stop_loss_share": (trades.label == -1).mean(), "mean_trade_return_%": trades.ret.mean() * 100})
perf = pd.DataFrame(perf_rows).set_index("strategy")
trade_stats = pd.DataFrame(trade_stats).set_index("model")
base_tp_rate = test.drop_duplicates(["date", "ticker"])["label"].eq(1).mean()

In [ ]:
perf.round(3)

In [ ]:
print(f"take-profit share of ALL test events (base rate): {base_tp_rate:.3f}")
trade_stats.round(3)

**Answer:** no model beats buy-and-hold. Holding the 14 stocks returned 97 % (CAGR 27 %, Sharpe 1.06, max drawdown −28 %), while
the rules returned 1–21 % (Sharpe 0.10–0.78).

(!) Total returns are not a fair comparison: each stock gets 1/14 of the capital and is held only after a signal, so on average just
**5–11 % of the capital is invested** (the `exposure` column counts days with at least one open position, 39–71 %). Sharpe ratio
and drawdown ignore how much capital is used, and on Sharpe every rule is still below buy-and-hold (best: LSTM 0.78, logistic
regression 0.76, SimpleRNN 0.64 vs 1.06). The small drawdowns (−3 % to −10 %) mostly reflect the idle cash.

The signals do select better trades than average: 44–51 % of the rules' trades hit take-profit, against 34 % of all test events.
The validation-optimal thresholds did not carry over well (GRU: validation Sharpe 1.54, test 0.10).

# [7/9] Is it better than luck? Random-entry benchmark
> **Method:** for each model, enter on **random** days with the same number of signals per stock, using the same rule and costs,
> 200 times. If the model's Sharpe is not clearly above most random runs, its signals add nothing beyond being in the market.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
ref_model = BEST_VALID_MODEL
ref_sig = pd.Series(P["test"][ref_model][:, 2] >= best_tau.loc[ref_model, "tau"])
per_ticker_n = ref_sig.groupby(test["ticker"]).sum()
random_sharpes, random_curves = [], []
for _ in range(N_RANDOM):
    sig = pd.Series(False, index=test.index)
    for t, n in per_ticker_n.items():
        idx = test.index[test.ticker == t]
        sig[rng.choice(idx, int(n), replace=False)] = True
    r, _ = portfolio(test, sig, test_prices)
    r = r[(r.index > t_start) & (r.index <= t_end)]
    random_sharpes.append(performance(r)["sharpe"])
    random_curves.append(r.cumsum())
random_sharpes = np.array(random_sharpes)
model_sharpe = perf.loc[ref_model, "sharpe"]
percentile = (random_sharpes < model_sharpe).mean() * 100
curves["Random entries (median)"] = pd.concat(random_curves, axis=1).median(axis=1).diff().fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.6, 1]})
for name, r in curves.items():
    color = STRATEGY_COLORS.get(name)
    lw = 2.5 if name in (ref_model, "Buy & hold (same stocks)") else 1.2
    axes[0].plot(np.exp(r.cumsum()), label=name, color=color, lw=lw)
axes[0].set(title="Growth of $1 on the test period (after costs)", ylabel="equity")
axes[0].legend(fontsize=9)
axes[1].hist(random_sharpes, bins=30, color="#9ca3af")
axes[1].axvline(model_sharpe, color="#dc2626", lw=2.5, label=f"{ref_model}: {model_sharpe:.2f}")
axes[1].set(title=f"Sharpe of {N_RANDOM} random-entry runs", xlabel="Sharpe")
axes[1].legend()
savefig(fig, "05_backtest")
plt.close(fig)

In [ ]:
print(f"{ref_model} Sharpe {model_sharpe:.2f} beats {percentile:.0f} % of random-entry runs "
      f"(random median {np.median(random_sharpes):.2f}, 5-95 % range {np.percentile(random_sharpes, 5):.2f} to "
      f"{np.percentile(random_sharpes, 95):.2f})")

In [ ]:
fig

**Answer:** not convincingly. SimpleRNN's Sharpe of 0.64 beats 79 % of random-entry runs with the same number of trades per stock
(random median 0.27). That is better than typical luck but well inside the random range (5–95 %: −0.51 to 0.92), so this test
cannot distinguish the signal from chance.

# [8/9] Explainability: SHAP for LightGBM
> **Question:** which features push the LightGBM model towards **take-profit**?
> SHAP splits each prediction (log-odds of the class) into feature contributions $f(x) = \phi_0 + \sum_j \phi_j$.

In [ ]:
lgbm = joblib.load(MODELS_DIR / "lightgbm.joblib")
dataset = load_processed("dataset")
prep = json.load(open(REPORTS_DIR / "preprocessing.json"))
X_test = dataset[dataset.split == "test"][FEATURES]
X_test = np.clip((X_test - prep["scaler_mean"]) / prep["scaler_scale"], -prep["clip"], prep["clip"]).astype(np.float32)
X_shap = X_test.sample(SHAP_SAMPLE, random_state=RANDOM_STATE)
explanation = shap.TreeExplainer(lgbm)(X_shap)
tp_values = explanation[:, :, 2]                       # class index 2 = take-profit
importance = pd.Series(np.abs(tp_values.values).mean(0), index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(9, 7))
shap.plots.beeswarm(tp_values, max_display=15, show=False, plot_size=None)
plt.title("SHAP values for take-profit (LightGBM, test sample)")
fig_shap = plt.gcf()
savefig(fig_shap, "05_shap_take_profit")
plt.close(fig_shap)

In [ ]:
importance.round(4).head(10)

In [ ]:
fig_shap

**Answer:** `vol_21` dominates by far (mean |SHAP| 0.085, twice the next feature), followed by `ffd_close` (the fractionally
differenced price), `mkt_ret_5`, `atr_14` and `vol_ratio_5_21`. Mostly volatility and market context, very little momentum —
consistent with the confusion matrix: the model mainly predicts *how much* a stock will move.

# [9/9] Summary & limitations

In [ ]:
summary = {
    "test_log_loss": test_metrics["log_loss"].round(4).to_dict(),
    "test_balanced_acc": test_metrics["balanced_acc"].round(4).to_dict(),
    "backtest": perf[["sharpe", "cagr", "max_drawdown", "exposure"]].round(4).to_dict(orient="index"),
    "random_entry_percentile_of_" + ref_model: round(float(percentile), 1),
    "thresholds": best_tau["tau"].to_dict(),
}
json.dump(summary, open(REPORTS_DIR / "test_results.json", "w"), indent=2, default=float)

In [ ]:
pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)})

**Findings**
- Triple-barrier outcomes are predictable **slightly** better than chance on unseen 2024–2026 data (log-loss 1.060 vs 1.099,
  balanced accuracy 0.41 vs 0.33), and the edge held from validation to test.
- The models mainly predict **whether** a stock will move enough to hit a barrier, not the direction.
- The recurrent networks were not clearly better than logistic regression or LightGBM.
- As a long-only rule, no model beats buy-and-hold of the same stocks, and the best model's Sharpe sits inside the range of
  random entries. There is **no tradable edge** here, which is the expected result for liquid large-cap stocks with public data.

**Limitations**
- **Selection bias:** the 14 stocks were chosen in hindsight and mostly rose; buy-and-hold of them is an unusually strong benchmark.
- **One test period** (2024–2026, a strong market). Results could differ in other regimes; walk-forward retraining would test this.
- **Simple execution:** trades at the close, 10 bp per side, no slippage model, no shorting, no position sizing by probability.
- **Small effects:** feature–label relationships are weak (notebook 03), so differences between models are within noise.